Write a python dummy version of **RecursiveCharacterTextSplitter** which includes all the 3 heuristics

In [7]:
from typing import List

In [8]:
class DummyRecursiveCharacterTextSplitter:
    def __init__(
        self,
        chunk_size: int = 100,
        chunk_overlap: int = 20,
        separators: List[str] | None = None,
    ):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = separators or ["\n\n", "\n", " ", ""]

    def split_text(self, text: str) -> List[str]:
        """
        Entry point.
        """
        chunks = self._recursive_split(text, self.separators)

        # Hely overlap
        return self._apply_overlap(chunks)

    def _recursive_split(
        self,
        text: str,
        separators: List[str],
    ) -> List[str]:
        """
        Heuristic #1 and #2:
        - Use largest separator first.
        - If a piece is still too large, recurse using smaller separators.
        """
        if len(text) <= self.chunk_size:

          if not separators:
              return [
                  text[i : i + self.chunk_size]
                  for i in range(0, len(text), self.chunk_size)
              ]

        separator = separators[0]
        remaining = separators[1:]

        # Last fallback: raw character splitting
        if separator == "":
            return [
                text[i : i + self.chunk_size]
                for i in range(0, len(text), self.chunk_size)
            ]

        parts = text.split(separator)

        chunks = []
        current = ""

        for part in parts:
            candidate = (
                part if not current else current + separator + part
            )

            if len(candidate) <= self.chunk_size:
                current = candidate
            else:
                if current:
                    chunks.extend(
                        self._recursive_split(current, remaining)
                    )
                current = part

        if current:
            chunks.extend(
                self._recursive_split(current, remaining)
            )

        return chunks

    def _apply_overlap(self, chunks: List[str]) -> List[str]:
        """
        Heuristic #3:
        Add overlap from previous chunk.
        """
        if not chunks:
            return []

        result = [chunks[0]]

        for i in range(1, len(chunks)):
            # Get the overlapping part from the end of the previous chunk
            previous_chunk = chunks[i-1]
            # Ensure we don't try to take more characters than available in the previous chunk
            overlap_text = previous_chunk[-min(self.chunk_overlap, len(previous_chunk)):]

            # Prepend the overlap_text to the current chunk
            result.append(overlap_text + chunks[i])

        return result


# Example
if __name__ == "__main__":
    text = """
    Beneath the quiet vault of evening skies,
    where silver stars awaken one by one,
    the wandering river carries whispered dreams
    through meadows touched by moonlit dew.

    Along its banks the ancient willows bow,
    their branches trailing softly through the mist,
    as though they write forgotten histories
    upon the flowing pages of the night.

    Far in the distance, lonely mountains stand,
    their shadowed crowns embraced by drifting clouds.
    They keep the memory of a thousand storms,
    of countless dawns that painted them with gold,
    and countless sunsets fading into violet fire.

    A solitary traveler walks the winding road.
    His footsteps mingle with the song of crickets,
    and every mile unfolds a different world:
    a field of wildflowers swaying with the wind,
    a grove of pines murmuring in low voices,
    a valley where the morning gathers light.

    He pauses at a hill and lifts his gaze.
    Above him stretches the enduring sky,
    vast beyond measure, deep beyond thought,
    filled with silent constellations turning slowly
    through the endless architecture of time.

    The wind arrives from distant unknown seas.
    It carries traces of forgotten journeys,
    the scent of rain upon faraway forests,
    the echo of laughter from vanished villages,
    and melodies no instrument remembers.

    Night deepens. Yet the darkness is not empty.
    It glows with hidden life and quiet wonder.
    Fireflies drift like fragments of fallen stars,
    and somewhere in the trees an owl keeps watch,
    guardian of the hours before the dawn.

    At last the eastern horizon softens.
    The first pale threads of light appear.
    Shadows retreat across the sleeping earth,
    and every leaf becomes a mirror of morning.
    The river brightens, the mountains awaken,
    and the traveler continues on his way.

    For every ending yields another beginning,
    every road conceals another turning,
    and every heart that dares to wander far
    discovers worlds both vast and unexpected.

    Thus the day rises with gentle certainty,
    spilling gold across valleys and rivers,
    while the sky unfolds its boundless promise,
    and the journey, forever unfinished,
    carries onward beyond the edge of sight.
    """

    splitter = DummyRecursiveCharacterTextSplitter(
        chunk_size=80,
        chunk_overlap=10,
    )

    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks, 1):
        print(f"\n--- Chunk {i} ---")
        print(repr(chunk))


--- Chunk 1 ---
'Beneath the quiet vault of evening skies,'

--- Chunk 2 ---
'ing skies,where silver stars awaken one by one,'

--- Chunk 3 ---
'ne by one,the wandering river carries whispered dreams'

--- Chunk 4 ---
'red dreamsthrough meadows touched by moonlit dew.'

--- Chunk 5 ---
'onlit dew.Along its banks the ancient willows bow,'

--- Chunk 6 ---
'llows bow,their branches trailing softly through the mist,'

--- Chunk 7 ---
' the mist,as though they write forgotten histories'

--- Chunk 8 ---
' historiesupon the flowing pages of the night.'

--- Chunk 9 ---
'the night.Far in the distance, lonely mountains stand,'

--- Chunk 10 ---
'ins stand,their shadowed crowns embraced by drifting clouds.'

--- Chunk 11 ---
'ng clouds.They keep the memory of a thousand storms,'

--- Chunk 12 ---
'nd storms,of countless dawns that painted them with gold,'

--- Chunk 13 ---
'with gold,and countless sunsets fading into violet fire.'

--- Chunk 14 ---
'olet fire.A solitary traveler walks the win